In [6]:
import pandas as pd
from sqlalchemy import create_engine, text
from gerar_mapa import BASE


# ============================================================
# CONFIGURAÇÕES
# ============================================================

EMPRESAS= ["bram", "cbo", "starnav"]


for EMPRESA in EMPRESAS:
    
    ARQUIVO = BASE / "dados" / f"SURVEY_{EMPRESA}.xlsx"
    
    DB_USER = "postgres"
    DB_PASSWORD = "!Ademanda1456!"
    DB_HOST = "localhost"
    DB_PORT = "5432"
    DB_NAME = "ofi"

    SCHEMA_DESTINO = "raw"
    TABELA_DESTINO = f"posicoes_{EMPRESA}"

    TABELA_STAGING = f"staging_posicoes_{EMPRESA}"


    # ============================================================
    # CONEXÃO COM POSTGRESQL
    # ============================================================

    engine = create_engine(
        f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}"
        f"@{DB_HOST}:{DB_PORT}/{DB_NAME}"
    )


    # ============================================================
    # LER ABAS DO EXCEL
    # ============================================================

    excel = pd.ExcelFile(ARQUIVO)

    print("Abas encontradas:")

    for aba in excel.sheet_names:
        print(f" - {aba}")


    # ============================================================
    # PROCESSAR CADA EMBARCAÇÃO
    # ============================================================

    for aba in excel.sheet_names:

        print(f"\nProcessando: {aba}")

        # --------------------------------------------------------
        # Ler aba
        # --------------------------------------------------------

        df = pd.read_excel(
            ARQUIVO,
            sheet_name=aba
        )

        # --------------------------------------------------------
        # Verificar colunas obrigatórias
        # --------------------------------------------------------

        colunas_obrigatorias = [
            "data_consulta",
            "lat_a",
            "lon_a",
            "data_reportada",
            "status"
        ]

        faltantes = [
            coluna
            for coluna in colunas_obrigatorias
            if coluna not in df.columns
        ]

        if faltantes:

            print(
                f"ERRO: colunas ausentes: {faltantes}"
            )

            continue

        # --------------------------------------------------------
        # Selecionar somente os campos necessários
        # --------------------------------------------------------

        df = df[
            [
                "data_consulta",
                "lat_a",
                "lon_a",
                "data_reportada",
                "status"
            ]
        ].copy()

        # --------------------------------------------------------
        # Identificar embarcação
        # --------------------------------------------------------

        nome_embarcacao = aba.strip()

        with engine.connect() as conn:

            resultado = conn.execute(
                text("""
                    SELECT id_embarcacao
                    FROM core.embarcacoes
                    WHERE nome_embarcacao = :nome
                """),
                {
                    "nome": nome_embarcacao
                }
            ).fetchone()

        if resultado is None:

            print(
                f"AVISO: embarcação '{nome_embarcacao}' "
                f"não encontrada em core.embarcacoes."
            )

            continue

        id_embarcacao = resultado[0]

        print(
            f"Embarcação encontrada: "
            f"{nome_embarcacao} → ID {id_embarcacao}"
        )

        # --------------------------------------------------------
        # Padronizar data da consulta
        # --------------------------------------------------------

        df["data_consulta"] = pd.to_datetime(
            df["data_consulta"],
            errors="coerce"
        )

        # --------------------------------------------------------
        # Padronizar latitude
        # --------------------------------------------------------

        df["latitude"] = pd.to_numeric(
            df["lat_a"],
            errors="coerce"
        )

        # --------------------------------------------------------
        # Padronizar longitude
        # --------------------------------------------------------

        df["longitude"] = pd.to_numeric(
            df["lon_a"],
            errors="coerce"
        )

        # --------------------------------------------------------
        # Padronizar data reportada
        # --------------------------------------------------------

        df["data_reportada"] = pd.to_datetime(
            df["data_reportada"],
            errors="coerce"
        )

        # --------------------------------------------------------
        # Adicionar ID da embarcação
        # --------------------------------------------------------

        df["id_embarcacao"] = id_embarcacao

        # --------------------------------------------------------
        # Estrutura final
        # --------------------------------------------------------

        df_final = df[
            [
                "id_embarcacao",
                "data_consulta",
                "latitude",
                "longitude",
                "data_reportada",
                "status"
            ]
        ].copy()

        # --------------------------------------------------------
        # Remover registros inválidos
        # --------------------------------------------------------

        df_final = df_final.dropna(
            subset=[
                "data_consulta",
                "latitude",
                "longitude"
            ]
        )

        # --------------------------------------------------------
        # Remover duplicidades dentro do próprio Excel
        # --------------------------------------------------------

        df_final = df_final.drop_duplicates(
            subset=[
                "id_embarcacao",
                "data_consulta",
                "latitude",
                "longitude",
                "data_reportada",
                "status"
            ]
        )

        if df_final.empty:

            print("Nenhuma posição válida encontrada.")

            continue

        # ========================================================
        # STAGING
        # ========================================================

        df_final.to_sql(
            name=TABELA_STAGING,
            con=engine,
            schema=SCHEMA_DESTINO,
            if_exists="replace",
            index=False,
            method="multi"
        )

        # ========================================================
        # CARGA INCREMENTAL
        # ========================================================

        with engine.begin() as conn:

            resultado = conn.execute(
                text(f"""
                    INSERT INTO {SCHEMA_DESTINO}.{TABELA_DESTINO}
                    (
                        id_embarcacao,
                        data_consulta,
                        latitude,
                        longitude,
                        data_reportada,
                        status
                    )

                    SELECT
                        id_embarcacao,
                        data_consulta,
                        latitude,
                        longitude,
                        data_reportada,
                        status

                    FROM {SCHEMA_DESTINO}.{TABELA_STAGING}

                    ON CONFLICT (
                        id_embarcacao,
                        data_consulta,
                        latitude,
                        longitude,
                        data_reportada,
                        status
                    )

                    DO NOTHING
                """)
            )

            inseridos = resultado.rowcount

        print(
            f"{inseridos} novas posições inseridas."
        )


    # ============================================================
    # FINALIZAÇÃO
    # ============================================================

    with engine.begin() as conn:

        conn.execute(
            text(f"""
                DROP TABLE IF EXISTS
                {SCHEMA_DESTINO}.{TABELA_STAGING}
            """)
        )


    print("\nImportação incremental concluída.")


Abas encontradas:
 - BRAM ATLAS
 - BRAM BAHIA
 - BRAM BELEM
 - BRAM BRASIL
 - BRAM BRASILIA
 - BRAM BRAVO
 - BRAM BREEZE
 - BRAM BUCK
 - BRAM BUZIOS
 - BRAM HERO
 - BRAM POWER
 - BRAM RIO
 - BRAM SPIRIT
 - BRAM TITAN

Processando: BRAM ATLAS
Embarcação encontrada: BRAM ATLAS → ID 1
0 novas posições inseridas.

Processando: BRAM BAHIA
Embarcação encontrada: BRAM BAHIA → ID 2
0 novas posições inseridas.

Processando: BRAM BELEM
Embarcação encontrada: BRAM BELEM → ID 3
0 novas posições inseridas.

Processando: BRAM BRASIL
Embarcação encontrada: BRAM BRASIL → ID 4
0 novas posições inseridas.

Processando: BRAM BRASILIA
Embarcação encontrada: BRAM BRASILIA → ID 5
0 novas posições inseridas.

Processando: BRAM BRAVO
Embarcação encontrada: BRAM BRAVO → ID 6
0 novas posições inseridas.

Processando: BRAM BREEZE
Embarcação encontrada: BRAM BREEZE → ID 7
0 novas posições inseridas.

Processando: BRAM BUCK
Embarcação encontrada: BRAM BUCK → ID 8
0 novas posições inseridas.

Processando: BRAM BUZI

C:\Users\roger\AppData\Local\Temp\ipykernel_14724\1818533628.py:147: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["data_consulta"] = pd.to_datetime(


20 novas posições inseridas.

Processando: STARNAV PHOENIX
Embarcação encontrada: STARNAV PHOENIX → ID 49
20 novas posições inseridas.

Processando: STARNAV REGULUS
Embarcação encontrada: STARNAV REGULUS → ID 50
20 novas posições inseridas.

Processando: STARNAV TAURUS
Embarcação encontrada: STARNAV TAURUS → ID 51
20 novas posições inseridas.

Processando: STARNAV URSUS
Embarcação encontrada: STARNAV URSUS → ID 52
19 novas posições inseridas.

Processando: STARNAV VOLANS
Embarcação encontrada: STARNAV VOLANS → ID 53
20 novas posições inseridas.

Importação incremental concluída.


In [3]:
colunas = [
    "hora_consulta",
    "lat_a",
    "lon_a",
    "data_reportada",
    "hora_reportada",
    "status"
]

display(df[colunas].head(20))

print("\nNULOS:")
print(df[colunas].isna().sum())

,hora_consulta,lat_a,lon_a,data_reportada,hora_reportada,status
0,20:40:00,-22.880000,-43.130000,2026-07-16 20:40:00,20:40:00,Underway using Engine
1,15:15:00,-23.810000,-43.770000,2026-07-17 06:17:00,06:17:00,Underway using Engine
2,05:33:00,-24.703400,-42.409164,2026-07-21 05:33:00,05:33:00,ACTIVE
3,na,-24.771299,-42.018955,2026-07-21 19:27:00,na,Restricted Manoeuvrability
4,na,-24.700127,-42.407646,2026-07-22 05:17:00,na,Restricted Manoeuvrability
5,na,-24.703123,-42.408306,2026-07-22 20:37:00,na,Restricted Manoeuvrability
6,na,-24.703850,-42.405991,2026-07-23 08:28:00,na,Restricted Manoeuvrability
7,NaN,-24.703659,-42.405407,2026-07-23 11:32:00,NaN,Restricted Manoeuvrability
8,NaN,-24.703659,-42.405407,2026-07-23 11:32:00,NaN,Restricted Manoeuvrability
9,NaN,-24.703659,-42.405407,2026-07-23 11:32:00,NaN,Restricted Manoeuvrability



NULOS:
hora_consulta     12
lat_a              0
lon_a              0
data_reportada     0
hora_reportada    12
status             0
dtype: int64


In [2]:
import pandas as pd

ARQUIVO = BASE / "dados" / "SURVEY_BRAM.xlsx"

df = pd.read_excel(
    ARQUIVO,
    sheet_name="BRAM ATLAS"
)

print("DIMENSÕES:", df.shape)

print("\nCOLUNAS:")
print(df.columns.tolist())

print("\nPRIMEIRAS LINHAS:")
display(df.head(10))

print("\nTIPOS:")
print(df.dtypes)


DIMENSÕES: (19, 19)

COLUNAS:
['data_consulta', 'hora_consulta', 'partida', 'data_p', 'hora_p', 'lat_p', 'lon_p', 'chegada', 'data_c', 'hora_c', 'lat_c', 'lon_c', 'lat_a', 'lon_a', 'velocidade', 'rumo', 'data_reportada', 'hora_reportada', 'status']

PRIMEIRAS LINHAS:


,data_consulta,hora_consulta,partida,data_p,hora_p,lat_p,lon_p,chegada,data_c,hora_c,lat_c,lon_c,lat_a,lon_a,velocidade,rumo,data_reportada,hora_reportada,status
0,2026-07-16 20:40:00,20:40:00,na,na,na,na,na,na,na,na,na,na,-22.880000,-43.130000,10.0,166.0,2026-07-16 20:40:00,20:40:00,Underway using Engine
1,2026-07-17 15:15:00,15:15:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-23.810000,-43.770000,6.1,164.0,2026-07-17 06:17:00,06:17:00,Underway using Engine
2,2026-07-21 05:33:00,05:33:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-24.703400,-42.409164,0.0,0.0,2026-07-21 05:33:00,05:33:00,ACTIVE
3,2026-07-21 20:15:00,na,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-24.771299,-42.018955,NaN,NaN,2026-07-21 19:27:00,na,Restricted Manoeuvrability
4,2026-07-22 05:30:00,na,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-24.700127,-42.407646,NaN,NaN,2026-07-22 05:17:00,na,Restricted Manoeuvrability
5,2026-07-22 20:37:00,na,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-24.703123,-42.408306,NaN,NaN,2026-07-22 20:37:00,na,Restricted Manoeuvrability
6,2026-07-23 08:28:00,na,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-24.703850,-42.405991,NaN,NaN,2026-07-23 08:28:00,na,Restricted Manoeuvrability
7,2026-07-23 21:36:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-24.703659,-42.405407,NaN,NaN,2026-07-23 11:32:00,NaN,Restricted Manoeuvrability
8,2026-07-24 06:57:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-24.703659,-42.405407,NaN,NaN,2026-07-23 11:32:00,NaN,Restricted Manoeuvrability
9,2026-07-24 21:17:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-24.703659,-42.405407,NaN,NaN,2026-07-23 11:32:00,NaN,Restricted Manoeuvrability



TIPOS:
data_consulta     datetime64[ns]
hora_consulta             object
partida                   object
data_p                    object
hora_p                    object
lat_p                     object
lon_p                     object
chegada                   object
data_c                    object
hora_c                    object
lat_c                     object
lon_c                     object
lat_a                    float64
lon_a                    float64
velocidade               float64
rumo                     float64
data_reportada    datetime64[ns]
hora_reportada            object
status                    object
dtype: object
